
# Crypto Trading Strategy – The Overfitters

## Syfte

Syftet med denna notebook är att undersöka
om historiska marknadsmönster kan användas
för att identifiera potentiella köp- och
säljtillfällen för Ethereum.

Vi analyserar:

1. Långsiktiga pristrender med SMA 200.
2. Momentum genom RSI 14.
3. Bull Market Support Band (BMSB).
4. Historiska köp- och säljsignaler.
5. Kapitalutveckling genom backtesting.
6. Jämförelse mot Buy & Hold.
7. Möjligheten att kombinera teknisk analys
   med maskininlärning.

## Forskningsfråga

Kan en kombination av tekniska indikatorer
identifiera historiska köp- och säljtillfällen
som ger bättre riskjusterad avkastning än
en passiv Buy & Hold-strategi?

## Metod

Vi börjar med en enkel regelbaserad modell.

Vi utvärderar strategin på historiska data
med hänsyn till transaktionskostnader
och tidsordning.

Därefter undersöker vi om maskininlärning
kan bidra med ytterligare prognosförmåga.

Vi skiljer tydligt mellan historiska resultat
och prognoser som ännu inte har validerats.


In [2]:

# STEG 99 – Separata indikatorer för ETH och BTC

import pandas as pd
import numpy as np

def build_strategy_data(asset):

    price_col = f"{asset}_Price"
    rsi_col = f"{asset}_RSI_14"

    # Läs prisdata
    data = df_prices[
        ["time", price_col]
    ].copy()

    data = data.sort_values("time")

    # Behåll giltiga priser
    data[price_col] = pd.to_numeric(
        data[price_col],
        errors="coerce"
    )

    data = data[
        data[price_col] > 0
    ].copy()

    # SMA 200
    data["SMA_200"] = (
        data[price_col]
        .rolling(200, min_periods=200)
        .mean()
    )

    # RSI 14 för rätt kryptovaluta
    rsi_data = df_features[
        ["time", rsi_col]
    ].copy()

    data = data.merge(
        rsi_data,
        on="time",
        how="left",
        validate="one_to_one"
    )

    # Veckopriser
    weekly = (
        data.set_index("time")[price_col]
        .resample("W-SUN")
        .last()
        .dropna()
        .to_frame()
    )

    # Bull Market Support Band
    weekly["SMA_20W"] = (
        weekly[price_col]
        .rolling(20, min_periods=20)
        .mean()
    )

    weekly["EMA_21W"] = (
        weekly[price_col]
        .ewm(
            span=21,
            adjust=False,
            min_periods=21
        )
        .mean()
    )

    # Tillgänglig först efter avslutad vecka
    weekly.index = (
        weekly.index + pd.Timedelta(days=1)
    )

    weekly = weekly.reset_index().rename(
        columns={"time": "Available_From"}
    )

    # Koppla veckodata till dagliga priser
    data = pd.merge_asof(
        data.sort_values("time"),
        weekly[
            ["Available_From", "SMA_20W", "EMA_21W"]
        ].sort_values("Available_From"),
        left_on="time",
        right_on="Available_From",
        direction="backward"
    )

    # BMSB – övre och nedre gräns
    both_valid = data[
        ["SMA_20W", "EMA_21W"]
    ].notna().all(axis=1)

    data["BMSB_Upper"] = data[
        ["SMA_20W", "EMA_21W"]
    ].max(axis=1).where(both_valid)

    data["BMSB_Lower"] = data[
        ["SMA_20W", "EMA_21W"]
    ].min(axis=1).where(both_valid)

    return data.reset_index(drop=True)


# Skapa två separata dataset
df_eth_strategy = build_strategy_data("ETH")
df_btc_strategy = build_strategy_data("BTC")

# Kontroll
for asset, data in [
    ("ETH", df_eth_strategy),
    ("BTC", df_btc_strategy)
]:

    rsi_col = f"{asset}_RSI_14"

    required = [
        "SMA_200",
        rsi_col,
        "SMA_20W",
        "EMA_21W"
    ]

    complete = data.dropna(subset=required)

    print(f"\n{asset} – RESULTAT")
    print("Observationer:", len(data))
    print("Första prisdatum:", data["time"].min())
    print(
        "Första datum med alla indikatorer:",
        complete["time"].min()
    )
    print("\nSaknade indikatorvärden:")
    print(data[required].isna().sum())
    print("\nSenaste observationerna:")

    display(
        data[
            ["time", f"{asset}_Price"] + required
        ].tail(5)
    )



ETH – RESULTAT
Observationer: 4104
Första prisdatum: 2015-08-08 00:00:00+00:00
Första datum med alla indikatorer: 2016-02-23 00:00:00+00:00

Saknade indikatorvärden:
SMA_200       199
ETH_RSI_14     14
SMA_20W       135
EMA_21W       142
dtype: int64

Senaste observationerna:


,time,ETH_Price,SMA_200,ETH_RSI_14,SMA_20W,EMA_21W
4099,2026-10-28 00:00:00+00:00,2356.988509,2158.586902,39.319429,2163.381441,2319.184185
4100,2026-10-29 00:00:00+00:00,2256.031994,2158.916361,31.483320,2163.381441,2319.184185
4101,2026-10-30 00:00:00+00:00,2240.229980,2158.251587,30.460058,2163.381441,2319.184185
4102,2026-10-31 00:00:00+00:00,2252.073358,2157.895898,32.237692,2163.381441,2319.184185
4103,2026-11-01 00:00:00+00:00,2269.988406,2157.461958,34.946664,2163.381441,2319.184185



BTC – RESULTAT
Observationer: 4323
Första prisdatum: 2015-01-01 00:00:00+00:00
Första datum med alla indikatorer: 2015-07-19 00:00:00+00:00

Saknade indikatorvärden:
SMA_200       199
BTC_RSI_14     14
SMA_20W       137
EMA_21W       144
dtype: int64

Senaste observationerna:


,time,BTC_Price,SMA_200,BTC_RSI_14,SMA_20W,EMA_21W
4318,2026-10-28 00:00:00+00:00,81237.437826,73058.140475,47.630485,72353.029895,76542.155043
4319,2026-10-29 00:00:00+00:00,78193.426404,73095.680759,37.113106,72353.029895,76542.155043
4320,2026-10-30 00:00:00+00:00,77109.590747,73108.064985,34.216067,72353.029895,76542.155043
4321,2026-10-31 00:00:00+00:00,77401.017287,73124.202529,35.670152,72353.029895,76542.155043
4322,2026-11-01 00:00:00+00:00,78135.080289,73141.068107,39.309154,72353.029895,76542.155043



## Steg 100 – Köp- och säljsignaler

Vi skapar separata signaler för Ethereum och Bitcoin.

Signalerna baseras på tre tekniska indikatorer:

- SMA 200
- RSI 14
- Bull Market Support Band

Vi definierar tre möjliga signaler:

KÖP – Positiv långsiktig trend och momentum.

SÄLJ – Negativ långsiktig trend och momentum.

AVVAKTA – Villkoren för köp eller sälj är
inte uppfyllda.

Signalerna beräknas utifrån historiska
observationer utan framtida prisinformation.

Vi ska senare testa om signalerna historiskt
ger bättre resultat än Buy & Hold.


In [3]:

# Steg 100 – Skapa separata köp- och säljsignaler

import numpy as np
import pandas as pd

def create_trading_signals(data, asset):

    df = data.copy()

    price = f"{asset}_Price"
    rsi = f"{asset}_RSI_14"

    # Kontrollera att samtliga indikatorer finns
    required = [
        price,
        "SMA_200",
        rsi,
        "BMSB_Upper",
        "BMSB_Lower"
    ]

    valid = df[required].notna().all(axis=1)

    # Köp: positiv trend och RSI mellan 50–70
    buy = (
        valid
        & (df[price] > df["SMA_200"])
        & (df[price] > df["BMSB_Upper"])
        & df[rsi].between(50, 70)
    )

    # Sälj: negativ trend och RSI under 50
    sell = (
        valid
        & (df[price] < df["SMA_200"])
        & (df[price] < df["BMSB_Lower"])
        & (df[rsi] < 50)
    )

    # Klassificera marknadsläget
    df["Signal"] = np.select(
        [buy, sell],
        ["KÖP", "SÄLJ"],
        default="AVVAKTA"
    )

    # Saknade indikatorer är inte en avvaktasignal
    df.loc[~valid, "Signal"] = "INGEN DATA"

    return df


# Ethereum
df_eth_signals = create_trading_signals(
    df_eth_strategy,
    "ETH"
)

# Bitcoin
df_btc_signals = create_trading_signals(
    df_btc_strategy,
    "BTC"
)

# Kontrollera resultaten
for asset, data in [
    ("ETH", df_eth_signals),
    ("BTC", df_btc_signals)
]:

    print(f"\n===== {asset} =====")

    print("\nSignalfördelning:")
    print(data["Signal"].value_counts())

    print("\nSenaste 10 observationerna:")

    display(
        data[
            [
                "time",
                f"{asset}_Price",
                f"{asset}_RSI_14",
                "SMA_200",
                "BMSB_Upper",
                "BMSB_Lower",
                "Signal"
            ]
        ].tail(10)
    )



===== ETH =====

Signalfördelning:
Signal
AVVAKTA       1705
SÄLJ          1106
KÖP           1094
INGEN DATA     199
Name: count, dtype: int64

Senaste 10 observationerna:


,time,ETH_Price,ETH_RSI_14,SMA_200,BMSB_Upper,BMSB_Lower,Signal
4094,2026-10-23 00:00:00+00:00,2378.260399,38.142146,2155.883082,2318.522944,2131.194108,AVVAKTA
4095,2026-10-24 00:00:00+00:00,2351.985153,36.126949,2156.403483,2318.522944,2131.194108,AVVAKTA
4096,2026-10-25 00:00:00+00:00,2325.796597,34.188131,2157.089314,2318.522944,2131.194108,AVVAKTA
4097,2026-10-26 00:00:00+00:00,2316.598625,33.507959,2157.721054,2319.184185,2163.381441,AVVAKTA
4098,2026-10-27 00:00:00+00:00,2347.562225,37.981092,2158.236200,2319.184185,2163.381441,AVVAKTA
4099,2026-10-28 00:00:00+00:00,2356.988509,39.319429,2158.586902,2319.184185,2163.381441,AVVAKTA
4100,2026-10-29 00:00:00+00:00,2256.031994,31.483320,2158.916361,2319.184185,2163.381441,AVVAKTA
4101,2026-10-30 00:00:00+00:00,2240.229980,30.460058,2158.251587,2319.184185,2163.381441,AVVAKTA
4102,2026-10-31 00:00:00+00:00,2252.073358,32.237692,2157.895898,2319.184185,2163.381441,AVVAKTA
4103,2026-11-01 00:00:00+00:00,2269.988406,34.946664,2157.461958,2319.184185,2163.381441,AVVAKTA



===== BTC =====

Signalfördelning:
Signal
AVVAKTA       1805
KÖP           1287
SÄLJ          1032
INGEN DATA     199
Name: count, dtype: int64

Senaste 10 observationerna:


,time,BTC_Price,BTC_RSI_14,SMA_200,BMSB_Upper,BMSB_Lower,Signal
4313,2026-10-23 00:00:00+00:00,80924.183766,44.762499,72849.485045,76192.479967,71508.997304,AVVAKTA
4314,2026-10-24 00:00:00+00:00,80699.313127,43.993817,72892.695989,76192.479967,71508.997304,AVVAKTA
4315,2026-10-25 00:00:00+00:00,80038.905803,41.727512,72937.465226,76192.479967,71508.997304,AVVAKTA
4316,2026-10-26 00:00:00+00:00,79949.934329,41.417956,72978.271498,76542.155043,72353.029895,AVVAKTA
4317,2026-10-27 00:00:00+00:00,80800.537372,45.574922,73017.548832,76542.155043,72353.029895,AVVAKTA
4318,2026-10-28 00:00:00+00:00,81237.437826,47.630485,73058.140475,76542.155043,72353.029895,AVVAKTA
4319,2026-10-29 00:00:00+00:00,78193.426404,37.113106,73095.680759,76542.155043,72353.029895,AVVAKTA
4320,2026-10-30 00:00:00+00:00,77109.590747,34.216067,73108.064985,76542.155043,72353.029895,AVVAKTA
4321,2026-10-31 00:00:00+00:00,77401.017287,35.670152,73124.202529,76542.155043,72353.029895,AVVAKTA
4322,2026-11-01 00:00:00+00:00,78135.080289,39.309154,73141.068107,76542.155043,72353.029895,AVVAKTA


In [1]:

# STEG 101 – Tabeller och diagram för ETH och BTC

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

def visualize_trading_signals(data, asset):

    df = data.copy().sort_values("time")

    price_col = f"{asset}_Price"
    rsi_col = f"{asset}_RSI_14"

    print(f"\n===== {asset} – HANDELSANALYS =====")

    # 1. Signalfördelning
    summary = (
        df["Signal"]
        .value_counts()
        .rename_axis("Signal")
        .reset_index(name="Antal dagar")
    )

    summary["Andel (%)"] = (
        summary["Antal dagar"] / len(df) * 100
    ).round(2)

    display(summary)

    # 2. Prisdiagram med trendindikatorer
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df[price_col],
        name=f"{asset} – Pris",
        mode="lines",
        line=dict(color="#5DADE2", width=1.5)
    ))

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["SMA_200"],
        name="SMA 200",
        mode="lines",
        line=dict(color="#F39C12", width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Upper"],
        mode="lines",
        line=dict(width=0),
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Lower"],
        name="Bull Market Support Band",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(46,204,113,0.18)",
        line=dict(width=0)
    ))

    # Visa endast när en NY signal uppstår
    previous_signal = df["Signal"].shift(1)

    buy = df[
        (df["Signal"] == "KÖP")
        & (previous_signal != "KÖP")
    ]

    sell = df[
        (df["Signal"] == "SÄLJ")
        & (previous_signal != "SÄLJ")
    ]

    fig.add_trace(go.Scatter(
        x=buy["time"],
        y=buy[price_col],
        mode="markers",
        name="Köpsignal",
        marker=dict(
            symbol="triangle-up",
            size=11,
            color="#2ECC71"
        )
    ))

    fig.add_trace(go.Scatter(
        x=sell["time"],
        y=sell[price_col],
        mode="markers",
        name="Säljsignal",
        marker=dict(
            symbol="triangle-down",
            size=11,
            color="#FF4B4B"
        )
    ))

    fig.update_layout(
        title=f"{asset} – Pris och köp-/säljsignaler",
        xaxis_title="Datum",
        yaxis_title="Pris (USD)",
        template="plotly_dark",
        height=650,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)

    fig.show()

    # 3. Signalfördelning som stapeldiagram
    fig_signals = px.bar(
        summary,
        x="Signal",
        y="Antal dagar",
        color="Signal",
        text="Antal dagar",
        title=f"{asset} – Fördelning av handelssignaler",
        template="plotly_dark",
        color_discrete_map={
            "KÖP": "#2ECC71",
            "SÄLJ": "#FF4B4B",
            "AVVAKTA": "#F39C12",
            "INGEN DATA": "#7F8C8D"
        }
    )

    fig_signals.show()

    # 4. RSI-diagram
    fig_rsi = go.Figure()

    fig_rsi.add_trace(go.Scatter(
        x=df["time"],
        y=df[rsi_col],
        mode="lines",
        name="RSI 14",
        line=dict(color="#BB86FC")
    ))

    fig_rsi.add_hline(
        y=70,
        line_dash="dash",
        line_color="red"
    )

    fig_rsi.add_hline(
        y=30,
        line_dash="dash",
        line_color="green"
    )

    fig_rsi.update_layout(
        title=f"{asset} – RSI 14",
        xaxis_title="Datum",
        yaxis_title="RSI",
        yaxis_range=[0, 100],
        template="plotly_dark",
        height=450
    )

    fig_rsi.show()

    # 5. Tabell över nya köp- och säljsignaler
    trade_signals = df[
        (
            (df["Signal"] == "KÖP")
            & (previous_signal != "KÖP")
        )
        |
        (
            (df["Signal"] == "SÄLJ")
            & (previous_signal != "SÄLJ")
        )
    ]

    print(f"\n{asset} – Senaste 15 nya signalerna:")

    display(
        trade_signals[
            ["time", price_col, rsi_col, "Signal"]
        ].tail(15)
    )


# Ethereum – separat analys
visualize_trading_signals(
    df_eth_signals,
    "ETH"
)

# Bitcoin – separat analys
visualize_trading_signals(
    df_btc_signals,
    "BTC"
)


NameError: name 'df_eth_signals' is not defined

In [2]:
import sys
import nbformat
import plotly.io as pio

print("Python:", sys.executable)
print("nbformat:", nbformat.__version__)

pio.renderers.default = "browser"

ModuleNotFoundError: No module named 'nbformat'